# TP 4 — Corrélation et régression linéaire : mesurer et modéliser une relation
**Analyse de données — Master IA / Master RSD** · Chapitre 3 : Analyse statistique (corrélation, régression linéaire, interprétation)

*Mesurer le sens et la force d'une relation, l'ajuster par une droite, et juger honnêtement ce que le modèle explique.*

**Objectifs du TP**
1. **Calculer et interpréter** le coefficient de corrélation de Pearson $r$ (signe, force).
2. **Ajuster** une régression linéaire simple : à la main (moindres carrés), puis avec `LinearRegression`.
3. **Évaluer** le modèle avec $R^2$, l'utiliser pour **prédire** — sans extrapoler.
4. **Reconnaître les pièges** de $r$ : relation non linéaire, valeur aberrante, corrélation ≠ causalité.
5. **Lire une matrice de corrélation** sur des données réalistes et repérer la multicolinéarité.

| Volume | Organisation | Données | Livrable |
|---|---|---|---|
| 2 séances de 2 h (4 h au total) | Binômes · Notebook Jupyter | `etude_notes.csv` (exemple du cours) · `clients_enrichi.csv` (TP 3) | Notebook complété + conclusions |

⭐ **Continuité :** `etude_notes.csv` est l'exemple « heures d'étude / note » du chapitre 3. `clients_enrichi.csv` est le fichier du **TP 3** (988 clients, nettoyé et enrichi).

**Mode d'emploi** : complétez les cellules marquées `# TODO`, puis exécutez la cellule **✅ Auto-correction** qui suit. Répondez aux questions dans les cellules **✍️**.

---
## Mise en place

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

---
## Étape 1 — Mesurer une relation : la corrélation de Pearson
*La corrélation de Pearson, notée $r$, mesure le sens et la force d'une **relation linéaire** entre deux variables quantitatives* (section 1).

### 1.0 Charger le fichier
Huit étudiants : nombre d'**heures d'étude par semaine** et **note obtenue** (sur 20).

In [ ]:
notes = ...   # TODO : pd.read_csv("etude_notes.csv") — séparateur "," (celui par défaut)

print(notes.shape)
notes

✅ **Auto-correction**

In [ ]:
assert notes.shape == (8, 2), f"Attendu (8, 2), obtenu {notes.shape}"
assert list(notes.columns) == ["heures", "note"]
print("✅ 1.0 correct")

### 1.1 Toujours regarder d'abord : le nuage de points
Tracez la note ($Y$, variable cible) en fonction des heures d'étude ($X$, variable explicative).

In [ ]:
X = notes["heures"]   # variable explicative
y = notes["note"]     # variable cible

# TODO : nuage de points de y en fonction de X (plt.scatter), avec un titre et des noms d'axes

✍️ **Question :** avant tout calcul, que dit le graphique ? Sens de la relation ? Les points sont-ils à peu près alignés ?

…

### 1.2 Calculer $r$
$$r = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n} (x_i - \bar{x})^2}\;\sqrt{\sum_{i=1}^{n} (y_i - \bar{y})^2}}$$
1. Calculez `r_main` **à la main** avec la formule.
2. Calculez `r_pandas` avec `X.corr(y)` et vérifiez que vous obtenez la même chose.

In [ ]:
ex = X - X.mean()   # écarts à la moyenne de X
ey = y - y.mean()   # écarts à la moyenne de y

r_main = ...     # TODO : formule de Pearson avec ex et ey (.sum(), np.sqrt)
r_pandas = ...   # TODO : X.corr(y)

print(f"r (à la main) = {r_main:.3f} | r (pandas) = {r_pandas:.3f}")

✅ **Auto-correction**

In [ ]:
assert np.isclose(r_main, r_pandas), "la formule et .corr() doivent donner le même résultat"
assert round(r_main, 3) == 0.990, "r attendu : 0,990"
print("✅ 1.2 correct")

✍️ **Question :** interprétez $r$ avec l'échelle du cours ($|r| < 0{,}30$ : très faible · $0{,}30$ à $0{,}70$ : modérée · $> 0{,}70$ : forte). Que disent le **signe** et la **valeur absolue** ?

…

---
## Étape 2 — Régression linéaire simple (section 2)
On modélise la relation par une droite : $Y = \beta_0 + \beta_1 X + \varepsilon$.

### 2.1 Les moindres carrés ordinaires (MCO), à la main
$$\hat{\beta}_1 = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n} (x_i - \bar{x})^2} \qquad\qquad \hat{\beta}_0 = \bar{y} - \hat{\beta}_1\,\bar{x}$$
Calculez la pente `b1` et l'ordonnée à l'origine `b0` (réutilisez `ex` et `ey`).

In [ ]:
b1 = ...   # TODO : pente
b0 = ...   # TODO : ordonnée à l'origine

print(f"ŷ = {b0:.3f} + {b1:.3f} X")

✅ **Auto-correction**

In [ ]:
assert round(b1, 3) == 0.899, "pente attendue : 0,899"
assert round(b0, 3) == 4.786, "ordonnée à l'origine attendue : 4,786"
print("✅ 2.1 correct")

### 2.2 Avec scikit-learn
`LinearRegression()` ajuste la droite des moindres carrés. Ajustez le modèle puis comparez avec vos valeurs.

💡 scikit-learn attend un **tableau 2D** pour $X$ : utilisez le double crochet `notes[["heures"]]`.

In [ ]:
model = ...   # TODO : LinearRegression()
# TODO : ajuster le modèle (fit) sur notes[["heures"]] et y

a = model.intercept_   # ordonnée à l'origine
b = model.coef_[0]     # pente
print(f"ŷ = {a:.3f} + {b:.3f} X")

✅ **Auto-correction**

In [ ]:
assert np.isclose(a, b0) and np.isclose(b, b1), "scikit-learn et la formule MCO doivent donner la même droite"
print("✅ 2.2 correct")

✍️ **Question :** interprétez la **pente** et l'**ordonnée à l'origine** avec une phrase chacune, dans le contexte (heures, note).

…

### 2.3 Évaluer le modèle : $R^2$ (section 3)
$R^2$ = proportion de la variabilité de $Y$ expliquée par le modèle :
$$R^2 = 1 - \frac{\sum_{i} (y_i - \hat{y}_i)^2}{\sum_{i} (y_i - \bar{y})^2}$$
1. Calculez les prédictions `y_pred`, puis `r2` avec `model.score(...)`.
2. Recalculez-le **à la main** (`r2_main`) avec la formule.

In [ ]:
y_pred = ...    # TODO : model.predict(notes[["heures"]])
r2 = ...        # TODO : model.score(notes[["heures"]], y)
r2_main = ...   # TODO : formule ci-dessus

print(f"R² = {r2:.3f} | à la main = {r2_main:.3f} | r² = {r_main ** 2:.3f}")

✅ **Auto-correction**

In [ ]:
assert np.isclose(r2, r2_main), "model.score et la formule doivent donner le même R²"
assert np.isclose(r2, r_main ** 2), "en régression linéaire simple, R² = r²"
print("✅ 2.3 correct")

✍️ **Questions**
1. Interprétez la valeur de $R^2$ avec une phrase (« … % de … »).
2. Quel lien observez-vous entre $R^2$ et le $r$ de l'étape 1 ?

…

### 2.4 Visualiser : le nuage de points et la droite
*Toujours visualiser les données et la droite pour valider le modèle.* Tracez les points **et** la droite de régression sur le même graphique, avec une légende.

In [ ]:
# TODO : plt.scatter(X, y, ...) pour les données, plt.plot(X, y_pred, ...) pour la droite, puis plt.legend()

### 2.5 Prédire… sans extrapoler
Utilisez le modèle pour prédire la note d'étudiants qui travaillent 5 h, 10 h, 15 h et **25 h** par semaine.

In [ ]:
nouveaux = pd.DataFrame({"heures": [5, 10, 15, 25]})
nouveaux["note_predite"] = ...   # TODO : model.predict(nouveaux[["heures"]])
nouveaux.round(2)

✅ **Auto-correction**

In [ ]:
assert np.allclose(nouveaux["note_predite"], a + b * nouveaux["heures"]), "ŷ = a + b X"
assert nouveaux["note_predite"].iloc[-1] > 20, "regardez la prédiction pour 25 h..."
print("✅ 2.5 correct")

✍️ **Question :** que prédit le modèle pour 25 h ? Est-ce possible ? Quelle bonne pratique du cours a-t-on oubliée ?

…

### 2.6 Regarder les résidus
Le résidu d'un étudiant est l'écart entre sa note réelle et sa note prédite : $e_i = y_i - \hat{y}_i$. Si le modèle linéaire est adapté, les résidus sont répartis **au hasard** autour de 0 (hypothèses de linéarité et d'homoscédasticité). Cellule fournie.

In [ ]:
residus = y - y_pred

plt.figure(figsize=(7, 3))
plt.axhline(0, color="gray", linestyle="--")
plt.scatter(X, residus, color="crimson", s=60)
plt.xlabel("Heures d'étude par semaine (h)")
plt.ylabel("Résidu (points)")
plt.title("Résidus de la régression")
plt.show()

✍️ **Question :** les résidus sont-ils répartis au hasard autour de 0 ? Que révèle leur forme, malgré un $R^2$ de 0,98 ?

…

---
## Étape 3 — Les pièges du coefficient $r$
*« $r$ mesure uniquement une relation linéaire. […] Toujours analyser le contexte et les variables en jeu ! »*

### 3.1 Quatre jeux de données, les mêmes chiffres : le quartet d'Anscombe
Les quatre petits jeux de données ci-dessous (11 points chacun) sont célèbres en statistique. Pour chacun, calculez $r$, la pente, l'ordonnée à l'origine et $R^2$.

In [ ]:
x123 = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
anscombe = {
    "I":   (x123, [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    "II":  (x123, [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    "III": (x123, [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    "IV":  ([8] * 7 + [19] + [8] * 3, [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
}

In [ ]:
lignes = {}
for nom, (xs, ys) in anscombe.items():
    xs, ys = pd.Series(xs, name="x"), pd.Series(ys, name="y")
    m = LinearRegression().fit(xs.to_frame(), ys)
    lignes[nom] = {
        "r": ...,           # TODO : corrélation entre xs et ys
        "pente": ...,       # TODO : m.coef_[0]
        "intercept": ...,   # TODO
        "R2": ...,          # TODO : m.score(...)
    }

resume = pd.DataFrame(lignes).T.round(2)
resume

✅ **Auto-correction**

In [ ]:
assert (resume["r"] == 0.82).all(), "les quatre jeux ont le même r"
assert (resume["pente"] == 0.5).all() and (resume["intercept"] == 3.0).all(), "... et la même droite"
print("✅ 3.1 correct")

Cellule fournie : les quatre nuages de points, avec la droite $\hat{y} = 3 + 0{,}5X$.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharex=True, sharey=True)
for ax, (nom, (xs, ys)) in zip(axes, anscombe.items()):
    ax.scatter(xs, ys, color="royalblue")
    ax.plot([3, 20], [3 + 0.5 * 3, 3 + 0.5 * 20], color="crimson")
    ax.set_title(f"Jeu {nom} — r = {resume.loc[nom, 'r']}")
plt.tight_layout()
plt.show()

✍️ **Question :** pour chacun des quatre jeux, la droite de régression est-elle un bon modèle ? Que retenez-vous ?

…

### 3.2 $r \approx 0$ ne veut pas dire « aucune relation »
Ici $Y = X^2$ : $Y$ est **entièrement déterminé** par $X$. Calculez $r$.

In [ ]:
x_u = np.linspace(-3, 3, 61)
y_u = x_u ** 2

r_u = ...   # TODO : corrélation entre x_u et y_u — np.corrcoef(x_u, y_u)[0, 1]
print(f"r = {r_u:.3f}")

✅ **Auto-correction**

In [ ]:
assert abs(r_u) < 1e-9, "r doit être nul"
print("✅ 3.2 correct")

In [ ]:
plt.figure(figsize=(5, 3))
plt.scatter(x_u, y_u, s=12, color="royalblue")
plt.title(f"Y = X² — r = {abs(r_u):.2f}")
plt.show()

✍️ **Question :** comment une relation parfaite peut-elle donner $r = 0$ ?

…

---
## Étape 4 — Matrice de corrélation sur des données réalistes
*Quand plusieurs variables quantitatives sont étudiées simultanément, la matrice de corrélation permet de visualiser toutes les corrélations de Pearson par paires.*

### 4.0 Charger les clients du TP 3

In [ ]:
df = ...   # TODO : pd.read_csv("clients_enrichi.csv", ...) — séparateur ";"

print(df.shape)
df.head()

✅ **Auto-correction**

In [ ]:
assert df.shape == (988, 11), f"Attendu (988, 11), obtenu {df.shape} -> séparateur ?"
print("✅ 4.0 correct")

### 4.1 La matrice et sa carte de chaleur
1. Calculez la matrice de corrélation `corr` des cinq variables quantitatives.
2. Affichez-la avec `sns.heatmap` (`annot=True, fmt=".2f", cmap="RdYlGn", vmin=-1, vmax=1, center=0`).

💡 `df.corr(numeric_only=True)` garderait aussi la colonne `id`, qui est numérique… mais c'est un **identifiant** (TP 3) : on choisit les colonnes explicitement.

In [ ]:
num_cols = ["age", "revenu", "nb_enfants", "anciennete", "nb_achats"]

corr = ...   # TODO : matrice de corrélation de df[num_cols]

# TODO : carte de chaleur

✅ **Auto-correction**

In [ ]:
assert corr.shape == (5, 5), "5 variables quantitatives (sans id)"
assert np.allclose(np.diag(corr), 1), "la diagonale vaut 1"
assert np.allclose(corr, corr.T), "la matrice est symétrique"
print("✅ 4.1 correct")

### 4.2 Quelle est la relation la plus forte ?
*Se concentrer sur les corrélations élevées en valeur absolue.* Trouvez **par le code** la paire de variables la plus corrélée (hors diagonale).

In [ ]:
hors_diag = corr.where(~np.eye(len(corr), dtype=bool))   # fourni : la diagonale est remplacée par NaN

paire = ...   # TODO : hors_diag.abs().stack().idxmax() -> (variable 1, variable 2)
r_max = ...   # TODO : valeur de r pour cette paire (corr.loc[...])
print(paire, round(r_max, 2))

✅ **Auto-correction**

In [ ]:
assert set(paire) == {"revenu", "nb_achats"}
assert round(r_max, 2) == 0.29
print("✅ 4.2 correct")

✍️ **Question :** qualifiez cette relation avec l'échelle du cours. Que dire des autres paires ? Comparez avec l'exemple heures / note.

…

### 4.3 Une seule valeur aberrante suffit
*Attention aux valeurs aberrantes qui peuvent influencer $r$.* Au TP 2, on a corrigé un revenu saisi à **999 999**. Remettez cette valeur pour **un seul** client (sur 988) et recalculez la corrélation revenu – nombre d'achats.

In [ ]:
revenu_outlier = df["revenu"].copy()
revenu_outlier.iloc[0] = 999999

r_sans = ...   # TODO : corrélation entre df["revenu"] et df["nb_achats"]
r_avec = ...   # TODO : corrélation entre revenu_outlier et df["nb_achats"]
print(f"sans outlier : r = {r_sans:.2f} | avec outlier : r = {r_avec:.2f}")

✅ **Auto-correction**

In [ ]:
assert np.isclose(r_sans, r_max)
assert r_avec < r_sans - 0.1, "un seul point doit nettement faire baisser r"
print("✅ 4.3 correct")

✍️ **Question :** de combien $r$ change-t-il ? Quelle étape doit donc toujours précéder une analyse de corrélation ?

…

### 4.4 Multicolinéarité et causalité
On ajoute deux variables vues au TP 3 : le **logarithme** du revenu et le **niveau d'étude encodé** (Licence = 0, Master = 1, Doctorat = 2). Calculez la matrice de corrélation des quatre variables ci-dessous.

In [ ]:
df["log_revenu"] = np.log(df["revenu"])
df["niveau_code"] = df["niveau_etude"].map({"Licence": 0, "Master": 1, "Doctorat": 2})   # encodage ordinal (TP 3)

corr2 = ...   # TODO : matrice de corrélation de ["revenu", "log_revenu", "niveau_code", "nb_achats"]
corr2.round(2)

✅ **Auto-correction**

In [ ]:
assert corr2.shape == (4, 4)
assert corr2.loc["revenu", "log_revenu"] > 0.8, "revenu et log_revenu sont presque la même information"
assert 0.5 < corr2.loc["revenu", "niveau_code"] < 0.7
print("✅ 4.4 correct")

✍️ **Questions**
1. Quelle paire dépasse le seuil $|r| > 0{,}8$ du cours ? Que faut-il en faire dans un même modèle ?
2. Le niveau d'étude et le revenu sont corrélés ($r \approx 0{,}62$). Peut-on conclure qu'**obtenir un doctorat fait augmenter le revenu** ? Proposez au moins une autre explication.

…

---
## Étape 5 — Une régression sur des données réalistes
Peut-on prédire le **nombre d'achats** d'un client à partir de son **revenu** ?

### 5.1 Ajuster et évaluer
Ajustez une régression linéaire simple `nb_achats ~ revenu` sur les 988 clients ; récupérez la pente et $R^2$.

In [ ]:
X_c = df[["revenu"]]
y_c = df["nb_achats"]

modele_c = ...   # TODO : LinearRegression ajustée sur (X_c, y_c)
pente_c = ...    # TODO : pente
r2_c = ...       # TODO : R²

print(f"achats = {modele_c.intercept_:.2f} + {pente_c:.7f} × revenu | R² = {r2_c:.3f}")

✅ **Auto-correction**

In [ ]:
assert np.isclose(r2_c, r_sans ** 2), "R² = r² (étape 4.3)"
assert 0 < pente_c * 10000 < 1, "combien d'achats en plus pour 10 000 DA de revenu en plus ?"
print("✅ 5.1 correct")

Cellule fournie : le nuage des 988 clients et la droite ajustée.

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.scatter(df["revenu"], y_c, alpha=0.25, s=15, color="royalblue", label="Clients")
plt.plot(df["revenu"], modele_c.predict(X_c), color="crimson", linewidth=2, label=f"Droite de régression (R² = {r2_c:.2f})")
plt.xlabel("Revenu (DA)")
plt.ylabel("Nombre d'achats")
plt.legend()
plt.show()

✍️ **Votre analyse**
1. La pente est minuscule : est-ce que cela veut dire que le revenu n'a aucun effet ? Exprimez-la pour **10 000 DA** de revenu supplémentaire.
2. Interprétez $R^2$. Ce modèle permet-il de prédire correctement le nombre d'achats **d'un client donné** ? Comparez avec l'étape 2.

⭐ **Bonus :** ajoutez `age`, `nb_enfants`, `anciennete` et `niveau_code` comme variables explicatives (régression **multiple** : `df[[...]]` avec plusieurs colonnes). $R^2$ augmente-t-il beaucoup ?

…

---
## Étape 6 — Synthèse et conclusions
*Objectif du chapitre : comprendre et quantifier la relation entre deux variables, puis interpréter les résultats obtenus avec Python.*

✍️ **Tableau de synthèse** — complétez avec vos résultats.

| Jeu de données | $r$ | Équation $\hat{y} = a + bX$ | $R^2$ | Interprétation |
|---|---|---|---|---|
| Heures d'étude → note | … | … | … | … |
| Revenu → nombre d'achats | … | … | … | … |

✍️ **Conclusions** (5 à 10 lignes) : que mesurent $r$ et $R^2$ ? Quels pièges avez-vous rencontrés ? Que faut-il toujours faire avant d'interpréter un coefficient ?

…

---
## Bilan

| Besoin | Outil | À retenir |
|---|---|---|
| Mesurer une relation linéaire | `s1.corr(s2)` | signe = sens, $|r|$ = force ; $-1 \le r \le 1$ |
| Toutes les paires d'un coup | `df[cols].corr()` + `sns.heatmap` | symétrique, diagonale = 1 ; $|r| > 0{,}8$ → multicolinéarité |
| Ajuster une droite | `LinearRegression().fit(X, y)` | `coef_[0]` = pente, `intercept_` = ordonnée à l'origine |
| Évaluer le modèle | `model.score(X, y)` | $R^2$ = part de variabilité expliquée ; $= r^2$ en régression simple |
| Prédire | `model.predict(X_new)` | ne pas extrapoler hors de la plage observée |
| Valider | nuage de points + résidus | toujours visualiser avant d'interpréter |

**Règle d'or :** corrélation ≠ causalité — et un chiffre ne remplace jamais un graphique.